## import dataset and preprocessing

In [ ]:
import intel_extension_for_pytorch as ipex

In [1]:
import torch
print(torch.__version__)
print("Environment working")

2.12.0+cpu
Environment working


In [3]:
import intel_extension_for_pytorch as ipex

ModuleNotFoundError: No module named 'intel_extension_for_pytorch'

In [1]:
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [53]:
data = pd.read_csv(r"./DataSet/IMDB Dataset.csv")

In [54]:
df = pd.DataFrame(data)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [55]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## preprocessing

### 1.convert to lower case

In [56]:
df["review"] = df["review"].str.lower()

### 2.remove Urls

In [57]:
import re


def remove_urls(text):
    text = re.sub(r"https\s+","",text)
    return text

df["review"] =df["review"].apply(remove_urls)

### 3.remove punnctuations (#,$,^,&,*,+,=,<>,!)

In [58]:
def remove_punnctuations(text):
    text = re.sub(r"^A-Za-z0-9\s","",text)
    return text

df["review"] =df["review"].apply(remove_punnctuations)

### 4.remove HTML

In [59]:
def remove_HTML(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] =df["review"].apply(remove_HTML)

### 5.remove the stop words (is, the, and ,a etc)

In [60]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sumit_mali\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sumit_mali\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sumit_mali\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [61]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [62]:
# sample data
sample_text = "This is a sample text for tokenization."
tokens = word_tokenize(sample_text)
print(tokens)

['This', 'is', 'a', 'sample', 'text', 'for', 'tokenization', '.']


In [63]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [64]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. filming technique u...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


### 6. Stemming

In [65]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [66]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [67]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,positive
1,wder ltle producti . film techniqu unssuming- ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,negative
4,petter mttei 's `` love time mey '' vulli stun...,positive


### 7. Encoding

In [68]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [69]:
y = df["sentiment"]

In [70]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [29]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,1
1,wder ltle producti . film techniqu unssuming- ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,0
4,petter mttei 's `` love time mey '' vulli stun...,1


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=1000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
X_train.shape

(39665, 1000)

In [33]:
X_test.shape

(9917, 1000)

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [35]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [36]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [37]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [39]:
import torch.nn as nn
import torch.optim as optim

In [71]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])

        return out

In [72]:
input_size = X_train.shape[1]

model = RNN(input_size)
model = model.to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [73]:
import torch

# Select device automatically
if torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print("Using:", device)

model = model.to(device)

Using: xpu


## Training the RNN

In [74]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
    
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.26383036375045776
epoch = 2/10 and loss = 0.40813973546028137
epoch = 3/10 and loss = 0.39450138807296753
epoch = 4/10 and loss = 0.3067679703235626
epoch = 5/10 and loss = 0.29091504216194153
epoch = 6/10 and loss = 0.2705813944339752
epoch = 7/10 and loss = 0.44011732935905457
epoch = 8/10 and loss = 0.39188259840011597
epoch = 9/10 and loss = 0.3651474416255951
epoch = 10/10 and loss = 0.2813868522644043


In [76]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        # Move test data to XPU
        Xb = Xb.to(device)
        yb = yb.to(device)

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)

        # move result back to CPU for .item()
        correct_vals += (predicted == yb).sum().cpu().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 83.26106685489563


In [50]:
print(next(model.parameters()).device)

xpu:0
